# SIMPLIcity: EfficientNet-B0 Optimization & Enhancement Suite
### Computer Vision Fine-Tuning with Aspect-Ratio Letterbox, Native 240x240 Resolution & Enhanced Head

This notebook trains and evaluates the optimized **EfficientNet-B0** pipeline on the SIMPLIcity dataset (1,000 images, 10 classes) using **Dual NVIDIA Tesla T4 GPUs** (	orch.nn.DataParallel).

---
### Key Optimizations Applied:
1. **Aspect-Ratio Preserving Letterbox**: Replaces direct center-cropping with symmetric padding to retain 100% of building spires and natural horizons.
2. **Native 240x240 Resolution**: Scales the feature map to  \times 8$ (64 spatial cells) matching EfficientNet-B0 compound scaling specification.
3. **Enhanced Head with BatchNorm1d**: Stabilizes feature scale and intermediate 256-dim projection (BatchNorm1d -> Dropout(0.3) -> Linear(1280, 256) -> SiLU -> BatchNorm1d -> Dropout(0.2) -> Linear(256, 10)).
4. **Two-Stage Fine-Tuning**: 5-epoch head warm-up with frozen backbone before opening differential LR fine-tuning ( \times 10^{-5}$ backbone vs  \times 10^{-4}$ head).
5. **Strict Deterministic Single-Pass Evaluation**: Evaluates purely on raw unaugmented test images without competition post-processing tricks.

## 1. Verify GPU Acceleration & Dual T4 Topology
Checks CUDA availability and device properties across both Tesla T4 GPUs.

In [ ]:
!nvidia-smi

import os

import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
count = torch.cuda.device_count()
print(f"GPU Count       : {count}")
for i in range(count):
    name = torch.cuda.get_device_name(i)
    props = torch.cuda.get_device_properties(i)
    mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / (1024 ** 3)
    print(f"  GPU {i}: {name} ({mem:.1f} GB VRAM)")
if count > 1:
    print(">> Dual GPU detected. DataParallel acceleration enabled across all cards <<")


## 2. Setup Environment & Sync Repository
Pulls the latest code updates from the GitHub repository.

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is in python path
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    if Path("/kaggle/working/cnn-SIMPLIcity").exists():
        repo_root = Path("/kaggle/working/cnn-SIMPLIcity")
    elif Path("cnn-SIMPLIcity").exists():
        repo_root = Path("cnn-SIMPLIcity").resolve()
    else:
        !git clone https://github.com/Arfiadi/cnn-SIMPLIcity.git
        repo_root = Path("cnn-SIMPLIcity").resolve()

os.chdir(str(repo_root))
!git pull origin main

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(f"Active working directory: {repo_root}")


## 3. Dataset Verification
Auto-detects and audits dataset split disjointness and file counts.

In [ ]:
!python scripts/audit_data.py


## 4. Inspect Enhanced EfficientNet-B0 Configuration
Displays the optimized parameters (240x240, Letterbox padding, Enhanced Head, Warm-up, TTA).

In [ ]:
from simplicity_cnn.config.schema import print_config_summary

cfg = print_config_summary("configs/efficientnet_b0.yaml")


## 5. Execute Training: Enhanced EfficientNet-B0
Trains EfficientNet-B0 with Two-Stage Fine-Tuning across Dual Tesla T4 GPUs.

In [ ]:
!python scripts/train.py --config configs/efficientnet_b0.yaml --multi-gpu --backbone-lr-ratio 0.1


## 6. Training Curves & Convergence Diagnostics
Plots loss convergence, accuracy progression, and confusion matrix.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

out_dir = Path("outputs/efficientnet_b0_enhanced_240")
summary_plot = out_dir / "gradient_descent_summary.png"
cm_plot = out_dir / "evaluation" / "confusion_matrix.png"

if summary_plot.exists():
    print("=== Training Summary Curves ===")
    display(Image(filename=str(summary_plot)))

if cm_plot.exists():
    print("=== Test Set Confusion Matrix ===")
    display(Image(filename=str(cm_plot)))


## 7. Performance Delta Analysis vs Baseline
Compares per-class F1 scores against the previous baseline (96.50%).

In [ ]:
import json
from pathlib import Path

import pandas as pd

report_file = Path("outputs/efficientnet_b0_enhanced_240/evaluation/classification_report.json")
if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        report = json.load(f)

    classes = ["People", "Beach", "Building", "Bus", "Dinosaur", "Elephant", "Flower", "Horse", "Mountain", "Food"]
    baseline_f1 = {
        "People": 0.9744, "Beach": 0.8889, "Building": 0.8889, "Bus": 1.0000,
        "Dinosaur": 1.0000, "Elephant": 1.0000, "Flower": 1.0000, "Horse": 1.0000,
        "Mountain": 0.9000, "Food": 1.0000
    }
    rows = []
    for c in classes:
        if c in report:
            prec = report[c]["precision"]
            rec = report[c]["recall"]
            f1 = report[c]["f1-score"]
            base = baseline_f1.get(c, 0.0)
            diff = (f1 - base) * 100
            rows.append({
                "Class": c,
                "Precision": f"{prec:.4f}",
                "Recall": f"{rec:.4f}",
                "F1-Score": f"{f1:.4f}",
                "Baseline F1": f"{base:.4f}",
                "Delta": f"{diff:+.2f}%"
            })
    df = pd.DataFrame(rows)
    print("=" * 75)
    print(" EFFICIENTNET-B0: PER-CLASS PERFORMANCE DELTA (ENHANCED VS BASELINE)")
    print("=" * 75)
    print(df.to_string(index=False))
    print("-" * 75)
    acc = report.get("accuracy", 0) * 100
    macro_f1 = report.get("macro avg", {}).get("f1-score", 0)
    print(f" Overall Enhanced Accuracy: {acc:.2f}% (Baseline: 96.50%)")
    print(f" Overall Enhanced Macro F1: {macro_f1:.4f} (Baseline: 0.9652)")
    print("=" * 75)


## 8. Misclassification Error Inspection
Inspects remaining test errors (if any) with confidence scores.

In [ ]:
import json
from pathlib import Path

err_file = Path("outputs/efficientnet_b0_enhanced_240/evaluation/misclassified_samples.json")
if err_file.exists():
    with open(err_file, "r", encoding="utf-8") as f:
        errors = json.load(f)
    print(f"Total Misclassified Samples: {len(errors)} / 200 (Previous Baseline: 7 errors)")
    for i, err in enumerate(errors, 1):
        fn = err['filename']
        t_lbl = err['true_label']
        p_lbl = err['predicted_label']
        c_val = err['confidence']
        print(f"  [{i:02d}] File: {fn} | True: {t_lbl:<10} -> Predicted: {p_lbl:<10} (Conf: {c_val:.4f})")
